# bayes-hdc quickstart

[bayes-hdc](https://github.com/rlogger/bayes-hdc) is a JAX library for hyperdimensional computing with statistical guarantees. In the next two minutes you will:

1. train an HDC classifier on a real dataset,
2. calibrate its probabilities and get conformal prediction sets with guaranteed coverage,
3. run a one-class anomaly detector with a guaranteed false-positive rate.

Runs on the free Colab CPU runtime.


In [ ]:
%pip install -q git+https://github.com/rlogger/bayes-hdc  # PyPI release imminent


## 1. Train a classifier

`HDClassifier` is scikit-learn compatible: it encodes feature vectors into 10,000-dimensional hypervectors and classifies with a centroid model.


In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X.astype(np.float32))
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
X_cal, X_test, y_cal, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=0, stratify=y_tmp)

from bayes_hdc.sklearn import HDClassifier

clf = HDClassifier(dimensions=10_000, encoder="kernel", gamma=0.003, random_state=0).fit(X_train, y_train)
print(f"accuracy: {(clf.predict(X_test) == y_test).mean():.3f}")


## 2. Calibrated probabilities and conformal sets

Raw similarity scores are badly calibrated. Temperature scaling fixes that, and a split-conformal wrapper turns the probabilities into prediction sets whose coverage is guaranteed to be at least `1 - alpha` (finite-sample, distribution-free).


In [ ]:
import jax.numpy as jnp
from bayes_hdc import TemperatureCalibrator, ConformalClassifier, expected_calibration_error

logits_cal  = jnp.log(clf.predict_proba(X_cal) + 1e-9)
logits_test = jnp.log(clf.predict_proba(X_test) + 1e-9)

calib = TemperatureCalibrator.create().fit(logits_cal, jnp.asarray(y_cal))
probs_cal, probs_test = calib.calibrate(logits_cal), calib.calibrate(logits_test)

ece_raw = expected_calibration_error(jnp.asarray(clf.predict_proba(X_test)), jnp.asarray(y_test))
ece_cal = expected_calibration_error(probs_test, jnp.asarray(y_test))
print(f"ECE before {ece_raw:.3f} -> after {ece_cal:.3f}")

conf = ConformalClassifier.create(alpha=0.1).fit(probs_cal, jnp.asarray(y_cal))
sets = np.asarray(conf.predict_set(probs_test))
print(f"coverage: {sets[np.arange(len(y_test)), y_test].mean():.3f}  (target >= 0.90)")
print(f"mean set size: {sets.sum(1).mean():.2f}")


## 3. Anomaly detection with a guaranteed false-positive rate

Fit on normal data only. `predict` flags outliers at a false-positive rate guaranteed to be at most `alpha` — no threshold tuning. The detector follows scikit-learn's `OutlierMixin` convention (+1 inlier / -1 outlier).


In [ ]:
from bayes_hdc.sklearn import HDAnomalyDetector

normal = X[y == 0]   # treat digit 0 as the normal class
anom   = X[y != 0]
norm_train, norm_test = train_test_split(normal, test_size=0.4, random_state=0)

det = HDAnomalyDetector(alpha=0.05, dimensions=10_000, random_state=0).fit(norm_train)
fpr = (det.predict(norm_test) == -1).mean()
hit = (det.predict(anom[:300]) == -1).mean()
print(f"false-positive rate on held-out normal: {fpr:.3f}  (guaranteed <= 0.05 in expectation)")
print(f"anomalies flagged: {hit:.3f}")


## Where to go next

- [Examples](https://github.com/rlogger/bayes-hdc/tree/main/examples) — 20 worked applications, from EMG gestures to conformal regression.
- [Benchmarks](https://github.com/rlogger/bayes-hdc/blob/main/BENCHMARKS.md) — honest multi-seed numbers vs TorchHD and sklearn baselines.
- [Documentation](https://rlogger.github.io/bayes-hdc/) — full API reference and design notes.

If this was useful, [a star on GitHub](https://github.com/rlogger/bayes-hdc) helps others find the library.
